In [ ]:
import copy, time
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA    = Path().resolve().parent.parent / 'data' / 'processed'
TS_BASE = ('https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/'
           'end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1/'
           'timeseries_individual_buildings/by_state/upgrade=0')

PARC      = 'nn_buildings_elargi.csv'
SEQ_LEN   = 168
STRIDE    = 168
N_OUT     = 4
CANAUX    = 64
DILATIONS = (1, 2, 4, 8, 16, 32)
EPOQUES   = 40
PATIENCE  = 5
GRAINE    = 42
NOMS      = ['total', 'chauffage', 'clim', 'eau_chaude']

RF = 1 + 4 * sum(DILATIONS)
print(f'champ réceptif théorique : {RF} h  (doit dépasser {SEQ_LEN})')

## L'architecture

```
 f(t)  (B, 168, 32)  --transpose-->  (B, 32, 168)
 s     (B, 51)       --tile-------->  (B, 51, 168)
                                          |
                                    concat (83 canaux)
                                          |
                              Conv1d 83 -> 64, noyau 1
                                          |
              6 blocs résiduels, dilatations 1, 2, 4, 8, 16, 32
                                          |
                              Conv1d 64 -> 4, noyau 1  + Softplus
                                          |
                       (B, 4, 168) --transpose--> (B, 168, 4)
```

- `padding='same'` symétrique → **non causal** : chaque heure voit son passé *et* son futur,
  comme le GRU bidirectionnel.
- Le vecteur statique est injecté **dès l'entrée** — et non seulement dans la tête — pour que
  les convolutions puissent moduler leur traitement selon le bâtiment à toutes les profondeurs.
  C'est la différence de fond avec le GRU, où `s` n'entre qu'après la récurrence.
- Les dilatations 1→32 donnent un champ réceptif de **253 h**, plus que la semaine : chaque
  heure prédite voit toute sa fenêtre. Ça couvre les constantes de temps thermiques du parc,
  mesurées de 4 h à 65 h.
- 64 canaux placent le budget à ~155 000 paramètres, soit celui du GRU bidirectionnel
  (164 356) : la comparaison porte sur l'architecture, pas sur la taille.

In [ ]:
class Bloc(nn.Module):
    """Bloc résiduel : deux convolutions dilatées, non causales."""

    def __init__(self, ch, dilation):
        super().__init__()
        self.c1 = nn.Conv1d(ch, ch, 3, padding='same', dilation=dilation)
        self.b1 = nn.BatchNorm1d(ch)
        self.c2 = nn.Conv1d(ch, ch, 3, padding='same', dilation=dilation)
        self.b2 = nn.BatchNorm1d(ch)
        self.act = nn.GELU()

    def forward(self, x):
        h = self.act(self.b1(self.c1(x)))
        h = self.b2(self.c2(h))
        return self.act(x + h)


class LoadConv(nn.Module):
    """ResNet convolutif dilaté : f(t) + statique -> conso(t), sortie contrainte >= 0."""

    def __init__(self, n_time, n_static, ch=CANAUX, n_out=N_OUT, dilations=DILATIONS):
        super().__init__()
        self.entree = nn.Conv1d(n_time + n_static, ch, 1)
        self.blocs  = nn.Sequential(*[Bloc(ch, d) for d in dilations])
        self.sortie = nn.Sequential(nn.Conv1d(ch, n_out, 1), nn.Softplus())

    def forward(self, x_time, static):
        x = x_time.transpose(1, 2)
        s = static.unsqueeze(-1).expand(-1, -1, x.size(-1))
        h = self.entree(torch.cat([x, s], dim=1))
        h = self.blocs(h)
        return self.sortie(h).transpose(1, 2)

In [ ]:
WEA = ['out.outdoor_air_drybulb_temp..c', 'out.outdoor_air_relative_humidity..percentage',
       'out.weather.wind_speed..meter_per_second',
       'out.weather.direct_normal_solar_radiation..watt_per_m2',
       'out.weather.diffuse_solar_radiation..watt_per_m2']
SCHED = ['out.schedules.' + s for s in [
    'occupants', 'vacancy', 'lighting_interior', 'lighting_garage', 'plug_loads_other',
    'plug_loads_tv', 'clothes_dryer', 'clothes_washer', 'dishwasher', 'cooking_range',
    'ceiling_fan', 'hot_water_fixtures', 'hot_water_clothes_washer', 'hot_water_dishwasher',
    'no_space_cooling', 'no_space_heating']]
SETP = ['out.schedules.heating_setpoint..c', 'out.schedules.cooling_setpoint..c']
TGT  = ['out.electricity.' + t + '.energy_consumption..kwh' for t in
        ['total', 'heating', 'cooling', 'hot_water']]


def load_building(bldg_id, state, cop_bat=1.0):
    path = DATA / f'{bldg_id}-0.parquet'
    ts = pd.read_parquet(path if path.exists() else f'{TS_BASE}/state={state}/{bldg_id}-0.parquet')
    if not path.exists():
        ts.to_parquet(path)
    ts['timestamp'] = pd.to_datetime(ts['timestamp'])
    brut = ts.set_index('timestamp').reindex(columns=WEA + SCHED + SETP + TGT)

    h = brut[WEA + SCHED + SETP].resample('1h').mean().iloc[:8760]
    y = brut[TGT].resample('1h').sum().iloc[:8760]
    h[SCHED] = h[SCHED].fillna(0.0)
    if h[SETP].isna().any().any():
        raise ValueError(f'bâtiment {bldg_id} : consignes absentes.')

    i = h.index
    cal = pd.DataFrame({
        'h_sin': np.sin(2*np.pi*i.hour/24),      'h_cos': np.cos(2*np.pi*i.hour/24),
        'd_sin': np.sin(2*np.pi*i.dayofweek/7),  'd_cos': np.cos(2*np.pi*i.dayofweek/7),
        'm_sin': np.sin(2*np.pi*(i.month-1)/12), 'm_cos': np.cos(2*np.pi*(i.month-1)/12),
    }, index=i)

    t_ext = h[WEA[0]]
    ecart = pd.DataFrame({
        'ecart_chauffage': (h[SETP[0]] - t_ext).clip(lower=0),
        'ecart_clim':      (t_ext - h[SETP[1]]).clip(lower=0),
    }, index=i)
    cop_t = np.clip(cop_bat * (0.6 + 0.02 * t_ext), 1.0, max(cop_bat, 1.0))
    ecart['besoin_elec_chauffage'] = ecart['ecart_chauffage'] / cop_t

    return pd.concat([h[WEA + SCHED + SETP], cal, ecart], axis=1), y

In [ ]:
BUILDINGS = list(pd.read_csv(DATA / PARC).itertuples(index=False, name=None))

_eff  = pd.read_parquet(DATA / 'metadata_clean.parquet',
                        columns=['bldg_id', 'in.hvac_heating_efficiency']).set_index('bldg_id')
_hspf = _eff['in.hvac_heating_efficiency'].astype(str).str.extract(r'([\d.]+)\s*HSPF')[0].astype(float)
COP   = (_hspf / 3.412).fillna(1.0).to_dict()

t0 = time.time()
F, YB, BIDS = [], [], []
for bid, st in BUILDINGS:
    f, y = load_building(bid, st, COP.get(bid, 1.0))
    F.append(f.values.astype('float32'))
    YB.append(y.values.astype('float32'))
    BIDS.append(bid)
F, YB, BIDS = np.stack(F), np.stack(YB), np.array(BIDS)
COLONNES = list(f.columns)

print(f'F {F.shape} | Y {YB.shape} | {time.time()-t0:.0f}s | '
      f'{F.nbytes/1e6:.0f} Mo + {YB.nbytes/1e6:.0f} Mo')

In [ ]:
preds = pd.read_parquet(DATA / 'static_preds_oos.parquet')
if 'bldg_id' in preds.columns:
    preds = preds.set_index('bldg_id')
assert list(preds.columns) == NOMS, list(preds.columns)

feat = pd.read_parquet(DATA / 'X_47features.parquet')
feat = feat.assign(tau=feat['C'] / (feat['UA'] + feat['H_ve']) / 3.6)
feat = feat.drop(columns=[c for c in feat.columns if 'setpoint' in c])
assert set(BIDS) <= set(feat.index)

equip = pd.read_parquet(DATA / 'metadata_clean.parquet',
                        columns=['bldg_id', 'in.hvac_cooling_type', 'in.water_heater_fuel',
                                 'in.hvac_heating_efficiency']).set_index('bldg_id')
a_clim = (equip.loc[BIDS, 'in.hvac_cooling_type'] != 'None').values.astype('float32')
ecs_el = (equip.loc[BIDS, 'in.water_heater_fuel'] == 'Electricity').values.astype('float32')
hspf = equip.loc[BIDS, 'in.hvac_heating_efficiency'].astype(str).str.extract(
    r'([\d.]+)\s*HSPF')[0].astype(float)
pac  = hspf.notna().values.astype('float32')
cop  = (hspf / 3.412).fillna(1.0).values.astype('float32')
mshp = equip.loc[BIDS, 'in.hvac_heating_efficiency'].astype(str).str.startswith(
    'MSHP').values.astype('float32')

S = np.hstack([preds.loc[BIDS].values, feat.loc[BIDS].values,
               a_clim[:, None], ecs_el[:, None],
               pac[:, None], cop[:, None], mshp[:, None]]).astype('float32')

un = np.ones_like(a_clim)
G  = np.stack([un, un, a_clim, ecs_el], axis=-1).astype('float32')

rng   = np.random.default_rng(GRAINE)
val_b = set(rng.choice(BIDS, size=max(1, round(0.2 * len(BIDS))), replace=False))
est_val = np.array([b in val_b for b in BIDS])
i_tr, i_va = np.where(~est_val)[0], np.where(est_val)[0]

xm, xs = F[i_tr].mean((0, 1), keepdims=True), F[i_tr].std((0, 1), keepdims=True) + 1e-8
sm, ss = S[i_tr].mean(0, keepdims=True),      S[i_tr].std(0, keepdims=True) + 1e-8
ys     = YB[i_tr].std((0, 1), keepdims=True) + 1e-8
Fn, Sn, Yn = (F - xm) / xs, (S - sm) / ss, YB / ys

print(f'{len(BIDS)} bâtiments | {len(i_tr)} entraînement | {len(i_va)} validation')
print(f'f(t) = {F.shape[-1]} entrées | s = {S.shape[1]} colonnes | min(Yn) = {Yn.min():.3f}')
print(f'sans clim : {int((a_clim == 0).sum())} bâtiments | sans ECS électrique : {int((ecs_el == 0).sum())}')

In [ ]:
class Fenetres(Dataset):
    def __init__(self, idx_bat, stride, L=SEQ_LEN):
        self.L = L
        self.pos = [(b, d) for b in idx_bat
                    for d in range(0, Fn.shape[1] - L + 1, stride)]

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, k):
        b, d = self.pos[k]
        sl = slice(d, d + self.L)
        return (torch.from_numpy(Fn[b][sl]), torch.from_numpy(Sn[b]),
                torch.from_numpy(Yn[b][sl]), torch.from_numpy(G[b]))


ds_tr = Fenetres(i_tr, STRIDE)
ds_va = Fenetres(i_va, SEQ_LEN)
print(f'entraînement : {len(ds_tr):,} fenêtres (stride {STRIDE})')
print(f'validation   : {len(ds_va):,} fenêtres (stride {SEQ_LEN}, disjointes)')

In [ ]:
_m = LoadConv(F.shape[-1], S.shape[1]).eval()
x = torch.zeros(1, SEQ_LEN, F.shape[-1], requires_grad=True)
s = torch.zeros(1, S.shape[1])
_m(x, s)[0, SEQ_LEN // 2].sum().backward()
touchees = (x.grad[0].abs().sum(-1) > 0).sum().item()

print(f'paramètres : {sum(p.numel() for p in _m.parameters()):,}')
print(f"l'heure {SEQ_LEN//2} dépend de {touchees} heures d'entrée sur {SEQ_LEN}")
print(f'champ réceptif {RF} h -> la semaine entière est couverte'
      if touchees == SEQ_LEN else 'ATTENTION : champ réceptif insuffisant')

In [ ]:
torch.manual_seed(GRAINE)
ld_tr = DataLoader(ds_tr, batch_size=64, shuffle=True,
                   generator=torch.Generator().manual_seed(GRAINE))
ld_va = DataLoader(ds_va, batch_size=128)
masque = lambda p, g: p * g.unsqueeze(1)

model = LoadConv(F.shape[-1], S.shape[1]).to(device)
opt, lossf = torch.optim.Adam(model.parameters(), lr=1e-3), nn.MSELoss()
print(f'{sum(p.numel() for p in model.parameters()):,} paramètres')

best, best_state, best_ep = float('inf'), None, -1
for epoch in range(EPOQUES):
    model.train(); t0 = time.time()
    for xb, sb, yb, gb in ld_tr:
        xb, sb, yb, gb = xb.to(device), sb.to(device), yb.to(device), gb.to(device)
        opt.zero_grad(); lossf(masque(model(xb, sb), gb), yb).backward(); opt.step()

    model.eval(); tot = n = 0
    with torch.no_grad():
        for xb, sb, yb, gb in ld_va:
            p = masque(model(xb.to(device), sb.to(device)), gb.to(device))
            tot += lossf(p, yb.to(device)).item() * len(xb); n += len(xb)
    vloss = tot / n
    print(f'epoch {epoch:3d}  val_loss {vloss:.4f}  ({time.time()-t0:.0f}s)')

    if vloss < best:
        best, best_ep, best_state = vloss, epoch, copy.deepcopy(model.state_dict())
    elif epoch - best_ep >= PATIENCE:
        break

model.load_state_dict(best_state)
print(f'meilleure val_loss {best:.4f} (époque {best_ep})')

In [ ]:
from sklearn.metrics import r2_score

model.eval(); P, T, GG = [], [], []
with torch.no_grad():
    for xb, sb, yb, gb in ld_va:
        P.append(masque(model(xb.to(device), sb.to(device)), gb.to(device)).cpu().numpy() * ys)
        T.append(yb.numpy() * ys); GG.append(gb.numpy())
pred, true, gate = np.concatenate(P), np.concatenate(T), np.concatenate(GG)

print('=== Validation (bâtiments jamais vus) ===')
for i, n in enumerate(NOMS):
    p, t = pred[..., i].ravel(), true[..., i].ravel()
    print(f'{n:12} R²={r2_score(t, p):6.3f}   RMSE={np.sqrt(((p-t)**2).mean()):.3f} kWh/h')

for i, n in [(2, 'clim'), (3, 'eau_chaude')]:
    absent = gate[:, i] == 0
    if absent.any():
        print(f'  {n:11} {int(absent.sum())} fenêtres sans équipement -> '
              f'max prédit = {pred[absent, :, i].max():.1e}')